https://365datascience.com/tutorials/python-tutorials/pca-k-means/

In [ ]:
# Load general libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import plotly.express as px

In [ ]:
df_segmentation = pd.read_csv('/app/data/cellprofiler_segmentation/watershed_nuclei.txt', sep='\t', index_col = 1)
pd.set_option('display.max_columns', None)

In [ ]:
display(df_segmentation)

In [ ]:
columns_to_drop = [range(7)]  # Posiciones de las columnas a eliminar
df_dropped = df_segmentation.drop(df_segmentation.columns[columns_to_drop], axis=1)
display(df_dropped)

In [ ]:
x_label = df_dropped.columns[0]
y_label = df_dropped.columns[3]

In [ ]:
x = df_dropped.iloc[:,0]
y = df_dropped.iloc[:,3]

In [ ]:
plt.figure()
plt.scatter(x, y)
plt.xlabel(x_label)
plt.ylabel(y_label)

# Añadir un título (opcional)
plt.title('Scatter Plot with Labels')

# Mostrar el gráfico
plt.show()

## PCA

In [ ]:
scaler = StandardScaler()
segmentation_std = scaler.fit_transform(df_dropped)

In [ ]:
pca = PCA()
pca.fit(segmentation_std)

In [ ]:
pca.explained_variance_ratio_

In [ ]:
plt.figure()

plt.plot(pca.explained_variance_ratio_.cumsum(), marker = 'o', color='black')
plt.xlabel('Number of components')
plt.ylabel('Explained variance')
plt.axhline(y=0.99, color='g', linestyle='--', linewidth=0.8, label='y = 0.99')
plt.axhline(y=0.95, color='orange', linestyle='--', linewidth=0.8, label='y = 0.8')
plt.axhline(y=0.8, color='r', linestyle='--', linewidth=0.8, label='y = 0.8')
plt.xlim(0, 8)
# Añadir un título (opcional)
plt.title('Explanied variance')

# Mostrar el gráfico
plt.show()

In [ ]:
pca = PCA(n_components = 2)

In [ ]:
pca.fit(segmentation_std)

## K-Means

In [ ]:
scores_pca = pca.transform(segmentation_std)
scores_pca

In [ ]:
wcss = []
for i in range (1,21):
    kmeans_pca = KMeans(n_clusters = i, init = 'k-means++', random_state = 42)
    kmeans_pca.fit(scores_pca)
    wcss.append(kmeans_pca.inertia_)

In [ ]:
plt.figure()

plt.plot(wcss, marker = 'o', color='black')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.title('K-means with PCA')

# Mostrar el gráfico
plt.show()

In [ ]:
kmeans_pca = KMeans(n_clusters = 4, init = 'k-means++', random_state = 42)

In [ ]:
kmeans_pca.fit(scores_pca)

In [ ]:
df_segm_pca_kmeans = pd.concat([df_dropped.reset_index(drop = True), pd.DataFrame(scores_pca)], axis = 1)
df_segm_pca_kmeans.columns.values[-3: ] = ['C1', 'C2', 'C3']
df_segm_pca_kmeans['Segment K-means PCA'] = kmeans_pca.labels_

In [ ]:
df_segm_pca_kmeans['Segment'] = df_segm_pca_kmeans['Segment K-means PCA'].map({
    0:'Primero',
    1:'Segundo',
    2:'Tercero',
    3:'Cuarto'})

In [ ]:
x_axis = df_segm_pca_kmeans['C1']
y_axis = df_segm_pca_kmeans['C2']

plt.figure()

sns.scatterplot(x=x_axis, y=y_axis, hue = df_segm_pca_kmeans['Segment'])
plt.show()

In [ ]:
import plotly.express as px

# Crear un DataFrame para Plotly
df_plotly = df_segm_pca_kmeans.copy()
df_plotly['Segment K-means PCA'] = df_plotly['Segment K-means PCA'].map({
    0: 'Primero',
    1: 'Segundo',
    2: 'Tercero'
})

# Crear el gráfico de dispersión 3D interactivo
fig = px.scatter_3d(df_plotly, x='C1', y='C2', z='C3', color='Segment K-means PCA',
                    labels={'C1': 'Componente 1', 'C2': 'Componente 2', 'C3': 'Componente 3'},
                    title='PCA with K-Means Clustering')

# Guardar el gráfico en un archivo HTML
fig.write_html('/app/pca_kmeans_3d.html')

print("El gráfico ha sido guardado como 'pca_kmeans_3d.html'. Puedes abrir este archivo en tu navegador para ver el gráfico interactivo.")